# 03 - Static Allocations

This notebook compares full-sample factor allocations. These results are descriptive and not used as walk-forward evidence.

## 1. Load data and settings

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
current_directory = Path.cwd()
repository_root = current_directory.parent if current_directory.name == "notebooks" else current_directory

if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from src import active_metrics, estimate_weights, load_config, load_monthly_returns, performance_metrics

In [ ]:
config = load_config(repository_root / "config" / "config.yaml")
monthly_returns = load_monthly_returns(repository_root / config["data_file"])
factor_columns = config["factor_columns"]
factor_returns = monthly_returns[factor_columns]

tables_directory = repository_root / "results" / "tables"
figures_directory = repository_root / "results" / "figures"
tables_directory.mkdir(parents=True, exist_ok=True)
figures_directory.mkdir(parents=True, exist_ok=True)

## 2. Calculate static weights

In [ ]:
methods = [
    "equal_weight",
    "inverse_volatility",
    "sample_gmv",
    "shrinkage_gmv",
    "erc"
]

static_weights = pd.DataFrame({
    method: estimate_weights(
        factor_returns,
        method=method,
        max_weight=config["maximum_weight"]
    )
    for method in methods
})

static_weights

In [ ]:
assert static_weights.index.tolist() == factor_columns
assert (static_weights >= 0).all().all()
assert (static_weights <= config["maximum_weight"] + 1e-10).all().all()
assert static_weights.sum().sub(1).abs().max() < 1e-9

## 3. Compare full-sample performance

In [ ]:
static_returns = pd.DataFrame({
    method: factor_returns.mul(static_weights[method], axis="columns").sum(axis=1)
    for method in methods
})
static_returns["market"] = monthly_returns[config["benchmark"]]

static_performance = static_returns.apply(performance_metrics).T
static_active = pd.DataFrame({
    method: active_metrics(static_returns[method], static_returns["market"])
    for method in methods
}).T
static_summary = static_performance.join(static_active)

static_summary

In [ ]:
static_weights.to_csv(tables_directory / "static_weights.csv")
static_summary.to_csv(tables_directory / "static_performance.csv")

In [ ]:
static_wealth = (1 + static_returns).cumprod()

fig, ax = plt.subplots(figsize=(12, 6))
static_wealth.plot(ax=ax, logy=True)
ax.set_title("Static full-sample allocations")
ax.set_xlabel("Date")
ax.set_ylabel("Wealth Index (Logarithmic)")
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(figures_directory / "static_wealth.png", dpi=150)
plt.show()

## Conclusion

The static comparison shows how the allocation rules behave over the full sample. Since every weight uses the same full history, the results remain descriptive.